# Notebook advantages
Has dbutils and spark
# Issues
Pathing is awful
  - EOE do have a function to help
  - DBX suggests wheels which is more technical
Hardcoding things dont want to so need to add them to some secrets
Spark may not run in pytest but is running here


In [0]:
import sys, os
# we need better routing
PROJECT_ROOT = "/Workspace/Users/philip.tate@nhs.net/PT Separate Feature Branch/src"
sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Contents:", os.listdir(PROJECT_ROOT))

from utils import load_csv_table


# Example configuration
layer = "bronze"
domain = "ods"
# had to do via command line locally need azure key vault access
storage_account_url = dbutils.secrets.get(scope="poc-secrets", key="storage_account_url")
base_path = f"abfss://{layer}{storage_account_url}/{domain}/"
csv_filename = "Contact_Details.csv"

# Load the CSV
#df = spark.read.option('header', 'true').csv(os.path.join(base_path, csv_filename))
df = load_csv_table(spark, base_path, csv_filename)
# Preview
df.show(5)


# Just better secret storage
  - # Example configuration
layer = "bronze"
domain = "ods"
# had to do via command line locally need azure key vault access
storage_account_url = dbutils.secrets.get(scope="poc-secrets", key="storage_account_url")
base_path = f"abfss://{layer}{storage_account_url}/{domain}/"
csv_filename = "Contact_Details.csv"

In [0]:
# List all secret scopes available in your workspace
# scopes = dbutils.secrets.listScopes()
# for scope in scopes:
#     print(scope.name)

# keys = dbutils.secrets.list("UnifiedReportingDevKeyVault")
# for key in keys:
#     print(key.key)

keys = dbutils.secrets.list("poc-secrets")
for key in keys:
    print(key.key)

# Test logic
First we need to just know can compare against a DF before we worry about test detection

Then we need pytest

In [0]:
"""Simple test for load_csv_table - Databricks UI version"""

from pyspark.sql import SparkSession
import sys

# Routing
PROJECT_ROOT = "/Workspace/Users/philip.tate@nhs.net/PT Separate Feature Branch/src"
sys.path.insert(0, PROJECT_ROOT)

from utils import load_csv_table


def test_load_csv_table():
    """Test that load_csv_table loads CSV correctly"""
    
    print("🧪 Starting test...")
    
    # Get Spark session
    spark = SparkSession.getActiveSession()
    
    # Create test data
    data = [
        ("ORG001", "Test Hospital", "Active"),
        ("ORG002", "Test Clinic", "Inactive"),
        ("ORG003", "Test Surgery", "Active")
    ]
    columns = ["OrganisationID", "Name", "Status"]
    
    test_df = spark.createDataFrame(data, columns)
    
    # Use DBFS temp location (works better in Databricks UI)
    import random
    test_id = random.randint(10000, 99999)
    temp_path = f"/tmp/test_csv_{test_id}"
    
    print(f"📁 Writing test data to: {temp_path}")
    
    try:
        # Write test data
        test_df.coalesce(1).write.csv(
            temp_path,
            header=True,
            mode="overwrite"
        )
        
        # Find the actual CSV file (Spark creates part-*.csv)
        files = dbutils.fs.ls(temp_path)
        csv_file = [f for f in files if f.name.startswith("part-") and f.name.endswith(".csv")][0]
        
        # Rename to expected filename
        new_path = f"{temp_path}/Contact_Details.csv"
        dbutils.fs.mv(csv_file.path, new_path)
        
        # TEST: Load the CSV
        base_path = temp_path + "/"
        csv_filename = "Contact_Details.csv"
        
        print(f"📂 Loading from: {base_path}{csv_filename}")
        df = load_csv_table(spark, base_path, csv_filename)
        
        # Verify results
        print(f"✓ DataFrame created: {df is not None}")
        print(f"✓ Row count: {df.count()} (expected 3)")
        print(f"✓ Columns: {df.columns}")
        
        rows = df.collect()
        print(f"✓ First row data: {rows[0]['OrganisationID']}, {rows[0]['Name']}, {rows[0]['Status']}")
        
        # Show the data
        print("\n📊 Loaded data:")
        df.show()
        
        # Assertions
        assert df is not None, "❌ DataFrame is None"
        assert df.count() == 3, f"❌ Expected 3 rows, got {df.count()}"
        assert "OrganisationID" in df.columns, "❌ Missing OrganisationID column"
        assert "Name" in df.columns, "❌ Missing Name column"
        assert "Status" in df.columns, "❌ Missing Status column"
        assert rows[0]["OrganisationID"] == "ORG001", "❌ Wrong data in first row"
        
        print("\n✅ ALL TESTS PASSED!")
        
        return df
        
    finally:
        # Clean up test data
        try:
            print(f"\n🧹 Cleaning up: {temp_path}")
            dbutils.fs.rm(temp_path, recurse=True)
            print("✓ Cleanup complete")
        except Exception as e:
            print(f"⚠️  Cleanup failed (not critical): {e}")


# 🚀 RUN THE TEST NOW
test_load_csv_table()